# Volcano Thesis — Results

InSAR volcanic deformation detection on Thalia `temporal/3`.

All numbers are parsed directly from the server training logs by `parse_logs.py`,
so nothing here is hand-transcribed.

**Run `python parse_logs.py` first** to regenerate `results_summary.json`.

In [ ]:
import json
import math
import statistics as st

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RESULTS = json.load(open('results_summary.json'))
print(f"{len(RESULTS)} runs loaded")

# Validated categorical palette (light mode); hues assigned in fixed order.
C = {'blue': '#2a78d6', 'orange': '#eb6834', 'aqua': '#1baf7a', 'yellow': '#eda100'}
INK, INK_MUTED, GRID, SURFACE = '#0b0b0b', '#52514e', '#e3e3e0', '#fcfcfb'

plt.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE,
    'axes.edgecolor': GRID, 'axes.labelcolor': INK_MUTED,
    'axes.titlecolor': INK, 'axes.grid': True, 'axes.axisbelow': True,
    'grid.color': GRID, 'grid.linewidth': 0.8,
    'xtick.color': INK_MUTED, 'ytick.color': INK_MUTED, 'text.color': INK,
    'font.size': 10, 'axes.titlesize': 12,
    'legend.frameon': False, 'lines.linewidth': 2,
})

def clean(ax):
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)
    return ax

## 1 — All runs

In [ ]:
rows = []
for r in RESULTS:
    c, t = r['config'], r['test']
    rows.append({
        'model': c.get('model', '?'), 'shuffled': bool(c.get('shuffle', False)),
        'channels': c.get('channels', 'all'), 'loss': c.get('loss', 'focal'),
        'wd': c.get('weight_decay', 1e-4), 'patience': c.get('patience', 0),
        'lr_sched': c.get('lr_schedule', 'cosine'), 'seed': c.get('seed', 42),
        'epochs': r['epochs_trained'], 'best_val_f1': r['best_val_f1'],
        'test_f1': t.get('f1'), 'test_prec': t.get('precision'),
        'test_rec': t.get('recall'), 'test_auroc': t.get('auroc'),
        'done': r['completed'], 'log': r['log_file'],
    })

df = pd.DataFrame(rows)
df_done = df[df['done']].reset_index(drop=True)
print(f"{len(df_done)} completed / {len(df)} total")
df_done.sort_values('test_f1', ascending=False)[
    ['model', 'shuffled', 'channels', 'loss', 'wd', 'patience',
     'lr_sched', 'seed', 'epochs', 'best_val_f1', 'test_f1', 'test_auroc']]

## 2 — Headline: our ConvLSTM vs the paper

Winning configuration (`--augment --loss ce --weight_decay 1e-2 --patience 20
--lr_schedule none`) at three seeds, against the paper's reported 3-seed
ConvLSTM with atmospheric channels.

In [ ]:
WINNING = dict(model='convlstm', shuffled=False, channels='all',
               loss='ce', wd=0.01, patience=20, lr_sched='none')

def select(d, **kw):
    m = pd.Series(True, index=d.index)
    for k, v in kw.items():
        m &= (d[k] == v)
    return d[m]

ordered  = select(df_done, **WINNING)
shuffled = select(df_done, **{**WINNING, 'shuffled': True})

def summarize(sub, name):
    out = {'condition': name, 'n': len(sub)}
    for m in ['test_f1', 'test_auroc', 'test_prec', 'test_rec']:
        v = sub[m].dropna().tolist()
        out[m] = f"{st.mean(v):.2f} +/- {st.stdev(v):.2f}" if len(v) > 1 else f"{v[0]:.2f}"
    return out

# Paper Table 3, time-series ConvLSTM with atmospheric channels
PAPER = {'f1': (78.89, 1.09), 'auroc': (96.19, 0.85),
         'prec': (77.01, 0.38), 'rec': (80.89, 2.21)}

pd.DataFrame([
    summarize(ordered,  'Ours - ordered'),
    summarize(shuffled, 'Ours - shuffled'),
    {'condition': 'Paper ConvLSTM (atm)', 'n': 3,
     'test_f1':    f"{PAPER['f1'][0]:.2f} +/- {PAPER['f1'][1]:.2f}",
     'test_auroc': f"{PAPER['auroc'][0]:.2f} +/- {PAPER['auroc'][1]:.2f}",
     'test_prec':  f"{PAPER['prec'][0]:.2f} +/- {PAPER['prec'][1]:.2f}",
     'test_rec':   f"{PAPER['rec'][0]:.2f} +/- {PAPER['rec'][1]:.2f}"},
])

In [ ]:
metrics = [('test_f1', 'f1', 'F1'), ('test_auroc', 'auroc', 'AUROC'),
           ('test_prec', 'prec', 'Precision'), ('test_rec', 'rec', 'Recall')]

ours_m  = [st.mean(ordered[k].dropna())  for k, _, _ in metrics]
ours_s  = [st.stdev(ordered[k].dropna()) for k, _, _ in metrics]
paper_m = [PAPER[p][0] for _, p, _ in metrics]
paper_s = [PAPER[p][1] for _, p, _ in metrics]
labels  = [lab for _, _, lab in metrics]

x, w = np.arange(len(labels)), 0.34
fig, ax = plt.subplots(figsize=(8, 4.2))
b1 = ax.bar(x - w/2 - 0.01, ours_m,  w, yerr=ours_s,  capsize=4,
            color=C['blue'],   label='Ours (ConvLSTM, 3 seeds)')
b2 = ax.bar(x + w/2 + 0.01, paper_m, w, yerr=paper_s, capsize=4,
            color=C['orange'], label='Paper ConvLSTM (3 seeds)')

for bars, means in ((b1, ours_m), (b2, paper_m)):
    for bar, m in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, m + 2.4, f'{m:.1f}',
                ha='center', va='bottom', fontsize=9, color=INK_MUTED)

ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel('%'); ax.set_ylim(0, 108)
ax.set_title('ConvLSTM: our reimplementation vs the paper (mean +/- std)', pad=12)
ax.legend(loc='lower right')
clean(ax); plt.tight_layout(); plt.show()

## 3 — Shuffled-frame ablation

Individual runs are plotted, not just means — the spread is the point.
The label is `any(frame_labels)`, **permutation-invariant by construction**,
so frame order cannot change any sample's correct answer.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
rng = np.random.default_rng(0)

for ax, metric, title in zip(axes, ['test_f1', 'test_auroc'],
                             ['Test F1', 'Test AUROC']):
    for i, (sub, name, col) in enumerate([(ordered, 'ordered', C['blue']),
                                          (shuffled, 'shuffled', C['orange'])]):
        v = sub[metric].dropna().to_numpy()
        ax.scatter(np.full(len(v), i) + rng.uniform(-0.07, 0.07, len(v)), v,
                   s=64, color=col, edgecolor=SURFACE, linewidth=2, zorder=3,
                   label=f'{name} (n={len(v)})')
        m, s = v.mean(), v.std(ddof=1)
        ax.hlines(m, i - 0.22, i + 0.22, color=col, linewidth=2.5, zorder=4)
        ax.vlines(i, m - s, m + s, color=col, linewidth=2, alpha=0.45, zorder=2)
        ax.text(i + 0.30, m, f'{m:.2f}\n+/- {s:.2f}', va='center',
                fontsize=9, color=INK_MUTED)
    ax.set_xticks([0, 1]); ax.set_xticklabels(['ordered', 'shuffled'])
    ax.set_xlim(-0.5, 1.7); ax.set_ylabel('%'); ax.set_title(title, pad=10)
    clean(ax)

axes[0].legend(loc='lower left')
fig.suptitle('Frame-order ablation - ranges overlap; no significant effect',
             y=1.00, fontsize=12, color=INK)
plt.tight_layout(); plt.show()

In [ ]:
a = ordered['test_f1'].dropna().to_numpy()
b = shuffled['test_f1'].dropna().to_numpy()
se = math.sqrt(a.var(ddof=1)/len(a) + b.var(ddof=1)/len(b))
t  = (a.mean() - b.mean()) / se

print(f"ordered   n={len(a)}  F1 {a.mean():.2f} +/- {a.std(ddof=1):.2f}   "
      f"range [{a.min():.2f}, {a.max():.2f}]")
print(f"shuffled  n={len(b)}  F1 {b.mean():.2f} +/- {b.std(ddof=1):.2f}   "
      f"range [{b.min():.2f}, {b.max():.2f}]")
print(f"\ndifference {a.mean()-b.mean():+.2f}pp   SE {se:.2f}   t = {t:.2f}")
print("best single run overall:",
      f"{'shuffled' if b.max() > a.max() else 'ordered'} at F1 {max(a.max(), b.max()):.2f}%")

## 4 — Same-seed reproducibility

Two shuffled runs were launched with identical arguments and the same (default)
seed 42, and did not produce the same result. `num_workers=4` webdataset
workers, `shardshuffle`, the albumentations RNG and CUDA non-determinism are
not covered by `torch.manual_seed`.

This measures irreducible run-to-run noise, which turns out to be **larger than
the between-seed spread** normally reported as error bars.

In [ ]:
same_seed = shuffled[shuffled['seed'] == 42]
if len(same_seed) > 1:
    print('Identical config, identical seed (42):\n')
    for _, r in same_seed.iterrows():
        print(f"  {r['log'][:64]:<66} F1 {r['test_f1']:6.2f}  AUROC {r['test_auroc']:6.2f}")
    print(f"\n  spread:  F1 {same_seed['test_f1'].max()-same_seed['test_f1'].min():.2f}pp"
          f"   AUROC {same_seed['test_auroc'].max()-same_seed['test_auroc'].min():.2f}pp")
    print(f"  between-seed std, ordered condition: "
          f"{ordered['test_f1'].std(ddof=1):.2f}pp")
else:
    print('Only one seed-42 run found.')

## 5 — How the ConvLSTM got there

The diagnostic sequence; each step fixed a specific identified failure.
Seed-42 single runs, shown to trace the progression.

In [ ]:
def one(**kw):
    s = select(df_done, model='convlstm', shuffled=False, channels='all', **kw)
    return None if s.empty else s.iloc[0]

steps = [
    ('focal, wd=1e-4\n(no aug)',   one(loss='focal', wd=0.0001, patience=0)),
    ('+ aug, CE, wd=1e-2',         one(loss='ce', wd=0.01, patience=0)),
    ('+ dropout,\npatience=10',    one(loss='focal', wd=0.01, patience=10)),
    ('+ CE,\npatience=20',         one(loss='ce', wd=0.01, patience=20, lr_sched='cosine')),
    ('+ fixed LR\n(winning)',      one(loss='ce', wd=0.01, patience=20,
                                       lr_sched='none', seed=42)),
]
steps = [(lab, r) for lab, r in steps if r is not None]

labels = [lab for lab, _ in steps]
f1s    = [r['test_f1'] for _, r in steps]
aurocs = [r['test_auroc'] for _, r in steps]
x = np.arange(len(labels))

fig, ax = plt.subplots(figsize=(9.5, 4.4))
ax.plot(x, f1s,    marker='o', markersize=9, color=C['blue'],
        markeredgecolor=SURFACE, markeredgewidth=2, label='Test F1')
ax.plot(x, aurocs, marker='o', markersize=9, color=C['aqua'],
        markeredgecolor=SURFACE, markeredgewidth=2, label='Test AUROC')
for xi, v in zip(x, f1s):
    ax.text(xi, v - 3.4, f'{v:.1f}', ha='center', fontsize=9, color=INK_MUTED)
for xi, v in zip(x, aurocs):
    ax.text(xi, v + 1.8, f'{v:.1f}', ha='center', fontsize=9, color=INK_MUTED)

ax.axhline(PAPER['f1'][0], color=INK_MUTED, linestyle=':', linewidth=1.5)
ax.text(len(labels) - 0.55, PAPER['f1'][0] + 1.0, "paper's F1 78.9",
        fontsize=9, color=INK_MUTED, ha='right')
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('%'); ax.set_ylim(60, 102)
ax.set_title('ConvLSTM diagnostic progression (seed 42)', pad=12)
ax.legend(loc='lower right')
clean(ax); plt.tight_layout(); plt.show()

## 6 — Architecture comparison at a matched recipe

All three architectures under the same training recipe, so the difference
is architectural rather than a hyperparameter artifact.

In [ ]:
matched = select(df_done, shuffled=False, channels='all', loss='ce',
                 wd=0.01, patience=20, lr_sched='cosine')
matched = matched[['model', 'test_f1', 'test_prec', 'test_rec', 'test_auroc']]
matched = matched.sort_values('test_f1', ascending=False).reset_index(drop=True)
print('Matched recipe: augment + CE + wd=1e-2 + patience=20 (cosine LR)\n')
display(matched)

fig, ax = plt.subplots(figsize=(7.5, 4))
bars = ax.bar(matched['model'], matched['test_f1'], 0.55, color=C['blue'])
for bar, v in zip(bars, matched['test_f1']):
    ax.text(bar.get_x() + bar.get_width()/2, v + 1.2, f'{v:.1f}',
            ha='center', fontsize=10, color=INK_MUTED)
ax.set_ylabel('Test F1 (%)'); ax.set_ylim(0, 92)
ax.set_title('Architecture, matched training recipe', pad=12)
clean(ax); plt.tight_layout(); plt.show()

## 7 — Learning curves

Early runs overfit hard (train loss to ~0 while validation degrades);
the fixes flatten that out.

In [ ]:
def history_of(log_name):
    for r in RESULTS:
        if r['log_file'] == log_name:
            return r
    return None

def find_log(**kw):
    s = select(df_done, **kw)
    return None if s.empty else s.iloc[0]['log']

curves = [
    ('ConvLSTM - winning recipe',
     find_log(model='convlstm', shuffled=False, loss='ce', wd=0.01,
              patience=20, lr_sched='none', seed=42), C['blue']),
    ('ConvLSTM - cosine LR',
     find_log(model='convlstm', shuffled=False, loss='ce', wd=0.01,
              patience=20, lr_sched='cosine'), C['orange']),
    ('ConvLSTM - no aug, focal (overfits)',
     find_log(model='convlstm', shuffled=False, loss='focal', wd=0.0001), C['aqua']),
]
curves = [(lab, lg, col) for lab, lg, col in curves if lg]

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for lab, lg, col in curves:
    h = history_of(lg)['history']
    ep = [e['epoch'] for e in h]
    axes[0].plot(ep, [e.get('val_f1') for e in h], color=col, label=lab)
    axes[1].plot(ep, [e.get('val_auroc') for e in h], color=col, label=lab)
    axes[2].plot(ep, [e.get('train_loss') for e in h], color=col, label=lab)

for ax, title, ylab in zip(axes, ['Validation F1', 'Validation AUROC', 'Train loss'],
                           ['%', '%', 'loss']):
    ax.set_title(title, pad=10); ax.set_xlabel('epoch'); ax.set_ylabel(ylab)
    clean(ax)
axes[2].set_yscale('log')
axes[0].legend(loc='lower right', fontsize=8)
plt.tight_layout(); plt.show()

## 8 — Atmospheric channel ablation

`--channels core` keeps only insar_difference / insar_coherence / DEM,
dropping the six atmospheric channels.

In [ ]:
atm  = select(df_done, model='convlstm', shuffled=False, loss='ce', wd=0.01,
              patience=20, lr_sched='none', seed=42)
core = atm[atm['channels'] == 'core']
allc = atm[atm['channels'] == 'all']

if not core.empty and not allc.empty:
    a_, c_ = allc.iloc[0], core.iloc[0]
    print(f"all channels (atm)  : F1 {a_['test_f1']:.2f}  AUROC {a_['test_auroc']:.2f}")
    print(f"core only    (no atm): F1 {c_['test_f1']:.2f}  AUROC {c_['test_auroc']:.2f}")
    print(f"delta                : F1 {c_['test_f1']-a_['test_f1']:+.2f}pp  "
          f"AUROC {c_['test_auroc']-a_['test_auroc']:+.2f}pp")
    print(f"paper's delta        : F1 {75.33-78.89:+.2f}pp  AUROC {92.12-96.19:+.2f}pp")
else:
    print('core-channel run not found')